In [2]:
import pathlib
from datetime import date, timedelta

import os
import sys
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import jqdatasdk as jq
import tushare as ts

In [3]:
print(pathlib.Path("."))
print(pathlib.Path(".").resolve())
print(pathlib.Path.cwd().resolve())

.
E:\Latitude_Analytics_v2\00_draft_collection_01
E:\Latitude_Analytics_v2\00_draft_collection_01


In [6]:
project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve() # Current Working Directory

for candidate_root in [current_path, * current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings

In [7]:
pro = ts.pro_api(settings.tushare_token)
jq.auth(settings.jqdata_id, settings.jqdata_secret)

auth success 


In [19]:
# =========================
# 2. 参数
# =========================

VARIETY = "RB"
START_DATE = "2026-01-01"
END_DATE = "2026-01-31"

DATA_ROOT = r"E:/Latitude_Analytics_v2/R00_draft_collection_01\duckdb_parquet_demo"
PARQUET_ROOT = os.path.join(DATA_ROOT, "fact_futures_daily")

In [21]:
# =========================
# 3. 拉取 rb 每日可交易合约日线
# =========================

trade_days = jq.get_trade_days(start_date=START_DATE, end_date=END_DATE)

all_rows = []

for trade_day in trade_days:
    trade_date = trade_day.strftime("%Y-%m-%d")

    # 某日 RB 品种可交易合约，例如 RB2601.XSGE
    contracts = jq.get_future_contracts(VARIETY, date=trade_date)

    print(trade_date, contracts)

    for jq_code in contracts:
        df = jq.get_price(
            jq_code,
            start_date=trade_date,
            end_date=trade_date,
            frequency="1d",
            fields=[
                "open",
                "high",
                "low",
                "close",
                "volume",
                "money",
                "open_interest",
            ],
            skip_paused=True,
            fq="none",
        )

        if df is None or df.empty:
            continue

        df = df.reset_index().rename(columns={"index": "datetime"})
        df["trade_date"] = trade_date
        df["jq_code"] = jq_code
        df["symbol"] = jq_code.split(".")[0]
        df["variety"] = VARIETY
        df["exchange"] = jq_code.split(".")[1]
        df["year"] = trade_date[:4]
        df["month"] = trade_date[5:7]

        all_rows.append(df)

daily_df = pd.concat(all_rows, ignore_index=True)
daily_df

2026-01-05 ['RB2601.XSGE', 'RB2602.XSGE', 'RB2603.XSGE', 'RB2604.XSGE', 'RB2605.XSGE', 'RB2606.XSGE', 'RB2607.XSGE', 'RB2608.XSGE', 'RB2609.XSGE', 'RB2610.XSGE', 'RB2611.XSGE', 'RB2612.XSGE']


E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickl

2026-01-06 ['RB2601.XSGE', 'RB2602.XSGE', 'RB2603.XSGE', 'RB2604.XSGE', 'RB2605.XSGE', 'RB2606.XSGE', 'RB2607.XSGE', 'RB2608.XSGE', 'RB2609.XSGE', 'RB2610.XSGE', 'RB2611.XSGE', 'RB2612.XSGE']


E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickl

2026-01-07 ['RB2601.XSGE', 'RB2602.XSGE', 'RB2603.XSGE', 'RB2604.XSGE', 'RB2605.XSGE', 'RB2606.XSGE', 'RB2607.XSGE', 'RB2608.XSGE', 'RB2609.XSGE', 'RB2610.XSGE', 'RB2611.XSGE', 'RB2612.XSGE']


...

2026-01-29 ['RB2602.XSGE', 'RB2603.XSGE', 'RB2604.XSGE', 'RB2605.XSGE', 'RB2606.XSGE', 'RB2607.XSGE', 'RB2608.XSGE', 'RB2609.XSGE', 'RB2610.XSGE', 'RB2611.XSGE', 'RB2612.XSGE', 'RB2701.XSGE']


E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickl

2026-01-30 ['RB2602.XSGE', 'RB2603.XSGE', 'RB2604.XSGE', 'RB2605.XSGE', 'RB2606.XSGE', 'RB2607.XSGE', 'RB2608.XSGE', 'RB2609.XSGE', 'RB2610.XSGE', 'RB2611.XSGE', 'RB2612.XSGE', 'RB2701.XSGE']


E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)
E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickl

,datetime,open,high,low,close,volume,money,open_interest,trade_date,jq_code,symbol,variety,exchange,year,month
0,2026-01-05,3105.0,3114.0,3074.0,3088.0,5906.0,1.825059e+08,16410.0,2026-01-05,RB2601.XSGE,RB2601,RB,XSGE,2026,01
1,2026-01-05,3122.0,3122.0,3080.0,3085.0,670.0,2.072924e+07,10487.0,2026-01-05,RB2602.XSGE,RB2602,RB,XSGE,2026,01
2,2026-01-05,3107.0,3107.0,3080.0,3089.0,33080.0,1.022793e+09,328112.0,2026-01-05,RB2603.XSGE,RB2603,RB,XSGE,2026,01
3,2026-01-05,3086.0,3109.0,3080.0,3086.0,527.0,1.629808e+07,18127.0,2026-01-05,RB2604.XSGE,RB2604,RB,XSGE,2026,01
4,2026-01-05,3135.0,3135.0,3097.0,3104.0,697016.0,2.170044e+10,1548351.0,2026-01-05,RB2605.XSGE,RB2605,RB,XSGE,2026,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,2026-01-30,3194.0,3212.0,3169.0,3172.0,2508.0,7.997508e+07,18634.0,2026-01-30,RB2609.XSGE,RB2609,RB,XSGE,2026,01
236,2026-01-30,3205.0,3220.0,3175.0,3177.0,42202.0,1.350478e+09,166767.0,2026-01-30,RB2610.XSGE,RB2610,RB,XSGE,2026,01
237,2026-01-30,3218.0,3229.0,3190.0,3198.0,111.0,3.568900e+06,314.0,2026-01-30,RB2611.XSGE,RB2611,RB,XSGE,2026,01
238,2026-01-30,3232.0,3245.0,3209.0,3210.0,422.0,1.362861e+07,558.0,2026-01-30,RB2612.XSGE,RB2612,RB,XSGE,2026,01


In [9]:
# =========================
# 4. 用 DuckDB 写 Hive 分区 Parquet
# =========================

connection = duckdb.connect()

connection.register("daily_df", daily_df)

connection.execute(f"""
COPY daily_df
TO '{PARQUET_ROOT}'
(
    FORMAT PARQUET,
    PARTITION_BY (variety, year, month),
    OVERWRITE_OR_IGNORE TRUE
)
""")

connection.close()

print(f"已写入: {PARQUET_ROOT}")

In [22]:
infos = jq.get_futures_info(["AG2012.XSGE",'A1609.XDCE' ])


In [23]:
infos

{'AG2012.XSGE': {'contract_multiplier': 15.0,
  'tick_size': 1.0,
  'trade_time': [['2019-12-17',
    '2020-12-15',
    '21:00~02:30',
    '09:00~10:15',
    '10:30~11:30',
    '13:30~15:00']]},
 'A1609.XDCE': {'contract_multiplier': 10.0,
  'tick_size': 1.0,
  'trade_time': [['2015-03-16',
    '2015-05-08',
    '21:00~02:30',
    '09:00~10:15',
    '10:30~11:30',
    '13:30~15:00'],
   ['2015-05-09',
    '2016-09-14',
    '21:00~23:30',
    '09:00~10:15',
    '10:30~11:30',
    '13:30~15:00']]}}

In [35]:
import re
from collections import defaultdict

import pandas as pd
from IPython.display import display, HTML


# =========================
# 0. 基础配置
# =========================

try:
    jq
except NameError:
    raise NameError("请先初始化聚宽对象 jq，例如 jq = ...")


BATCH_SIZE = 200

OUTPUT_HTML = "期货品种交易时间规则变动表_含先后顺序.html"


exchange_name_map = {
    "XSGE": "上期所",
    "XDCE": "大商所",
    "XZCE": "郑商所",
    "CCFX": "中金所",
    "XINE": "上期能源",
    "GFEX": "广期所",
    "XGFE": "广期所",
}


def parse_contract(code: str):
    """
    解析期货合约代码。

    AG2012.XSGE -> variety=AG, exchange=XSGE
    A1609.XDCE  -> variety=A,  exchange=XDCE
    IC1505.CCFX -> variety=IC, exchange=CCFX
    AG9999.XSGE -> variety=AG, exchange=XSGE
    """
    m = re.match(r"^([A-Z]+)\d+\.(\w+)$", code)

    if m is None:
        return None, None

    variety = m.group(1)
    exchange = m.group(2)

    return variety, exchange


def chunks(lst, n=200):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def normalize_sessions(row):
    """
    聚宽 trade_time 中一行的结构一般是：

    [
        start_date,
        end_date,
        session_1,
        session_2,
        ...
    ]

    这里只取 row[2:] 作为交易时段组合。
    """
    return tuple(row[2:])


def sessions_to_text(sessions):
    return " | ".join(sessions)


# =========================
# 1. 获取全部期货合约
# =========================

all_fut = jq.get_all_securities(types=["futures"])
codes = list(all_fut.index)

print(f"获取到期货合约数量: {len(codes)}")


# =========================
# 2. 批量获取合约信息
# =========================

all_infos = {}
failed_batches = []

for batch in chunks(codes, BATCH_SIZE):
    try:
        infos = jq.get_futures_info(batch)
        all_infos.update(infos)
    except Exception as e:
        failed_batches.append((batch, e))
        print("batch failed:", batch[:5], "...", e)

print(f"成功获取合约信息数量: {len(all_infos)}")
print(f"失败 batch 数量: {len(failed_batches)}")


# =========================
# 3. 拉平成 合约-交易时间规则段 明细表
# =========================

rule_rows = []

for code, info in all_infos.items():
    variety, exchange = parse_contract(code)

    if variety is None or exchange is None:
        continue

    trade_time = info.get("trade_time", [])

    for row in trade_time:
        if len(row) < 3:
            continue

        start_date = row[0]
        end_date = row[1]
        sessions_key = normalize_sessions(row)
        sessions_text = sessions_to_text(sessions_key)

        rule_rows.append({
            "contract": code,
            "exchange": exchange,
            "exchange_name": exchange_name_map.get(exchange, exchange),
            "variety": variety,
            "start_date": pd.to_datetime(start_date),
            "end_date": pd.to_datetime(end_date),
            "sessions_key": sessions_key,
            "sessions_text": sessions_text,
        })

rule_detail = pd.DataFrame(rule_rows)

if rule_detail.empty:
    raise ValueError("rule_detail 为空：没有解析到任何 trade_time 规则。")

print(f"合约-规则段明细行数: {len(rule_detail)}")


# =========================
# 4. 按 交易所 + 品种 + 交易时段组合 汇总
#    一行 = 某品种历史上出现过的一套交易时间规则
# =========================

rule_versions = (
    rule_detail
    .groupby(
        [
            "exchange",
            "exchange_name",
            "variety",
            "sessions_key",
            "sessions_text",
        ],
        as_index=False
    )
    .agg(
        first_start_date=("start_date", "min"),
        last_end_date=("end_date", "max"),
        n_contracts_with_rule=("contract", "nunique"),
        example_contracts=("contract", lambda x: list(pd.unique(x))[:8]),
    )
)


# 按首次出现时间排序，形成规则先后顺序
rule_versions = rule_versions.sort_values(
    [
        "exchange",
        "variety",
        "first_start_date",
        "last_end_date",
        "sessions_text",
    ]
).reset_index(drop=True)


rule_versions["rule_no"] = (
    rule_versions
    .groupby(["exchange", "variety"])
    .cumcount()
    + 1
)


rule_versions_long = rule_versions[
    [
        "exchange",
        "exchange_name",
        "variety",
        "rule_no",
        "first_start_date",
        "last_end_date",
        "sessions_text",
        "n_contracts_with_rule",
        "example_contracts",
    ]
].copy()


rule_versions_long["first_start_date"] = (
    rule_versions_long["first_start_date"]
    .dt.strftime("%Y-%m-%d")
)

rule_versions_long["last_end_date"] = (
    rule_versions_long["last_end_date"]
    .dt.strftime("%Y-%m-%d")
)


# =========================
# 5. 汇总成 一行一个品种 的总表
# =========================

def make_timeline_text(df):
    """
    把某个品种的多套交易时间规则，按 rule_no 拼成可读时间线。
    """
    df = df.sort_values("rule_no")

    lines = []

    for _, row in df.iterrows():
        line = (
            f"规则{row['rule_no']}："
            f"{row['first_start_date']} ~ {row['last_end_date']}；"
            f"{row['sessions_text']}"
        )
        lines.append(line)

    return "\n".join(lines)


summary_rows = []

for (exchange, variety), g in rule_versions_long.groupby(["exchange", "variety"]):
    g = g.sort_values("rule_no")

    n_session_rules = int(g["rule_no"].max())
    changed = n_session_rules > 1

    contracts_of_group = rule_detail.loc[
        (rule_detail["exchange"] == exchange) &
        (rule_detail["variety"] == variety),
        "contract"
    ]

    summary_rows.append({
        "exchange": exchange,
        "exchange_name": g["exchange_name"].iloc[0],
        "variety": variety,
        "changed_emoji": "🔴 已变动" if changed else "🟢 未变动",
        "changed": changed,
        "n_session_rules": n_session_rules,
        "n_contracts": contracts_of_group.nunique(),
        "rule_timeline_text": make_timeline_text(g),
    })

result_ordered = pd.DataFrame(summary_rows)

result_ordered = result_ordered.sort_values(
    ["changed", "exchange", "variety"],
    ascending=[False, True, True]
).reset_index(drop=True)

changed_ordered = result_ordered[result_ordered["changed"] == True].copy()
unchanged_ordered = result_ordered[result_ordered["changed"] == False].copy()


# =========================
# 6. HTML 展示函数：修正版
# =========================

import html
import pandas as pd
from pandas.api.types import is_scalar
from IPython.display import display, HTML


def cell_to_html(x):
    """
    把 DataFrame 单元格安全转换成 HTML 文本。
    能处理：
    - 普通字符串
    - 数值
    - NaN / None / NaT
    - list / tuple / set
    - dict
    """

    # 1. 处理 None
    if x is None:
        return ""

    # 2. 只对标量使用 pd.isna
    if is_scalar(x):
        try:
            if pd.isna(x):
                return ""
        except Exception:
            pass

    # 3. 处理 list / tuple / set
    if isinstance(x, (list, tuple, set)):
        text = ", ".join(str(v) for v in x)

    # 4. 处理 dict
    elif isinstance(x, dict):
        text = str(x)

    # 5. 处理其他对象
    else:
        text = str(x)

    # 6. HTML 转义，然后保留换行
    return html.escape(text).replace("\n", "<br>")


def prepare_html_df(df):
    """
    把 DataFrame 中的 object 列转换成适合 HTML 展示的字符串。
    """
    out = df.copy()

    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].map(cell_to_html)

    return out


def display_full_df(df, title, max_height=700, min_width=1500):
    show_df = prepare_html_df(df)

    html_table = show_df.to_html(
        index=False,
        escape=False,
    )

    html_text = f"""
    <h3>{html.escape(title)}</h3>
    <div style="
        max-height: {max_height}px;
        overflow: auto;
        border: 1px solid #555;
        padding: 8px;
        width: 100%;
        margin-bottom: 20px;
    ">
        <style>
            table {{
                border-collapse: collapse;
                font-size: 12px;
                min-width: {min_width}px;
            }}
            th, td {{
                border: 1px solid #555;
                padding: 4px 8px;
                vertical-align: top;
                white-space: normal;
            }}
            th {{
                position: sticky;
                top: 0;
                background-color: #222;
                color: white;
                z-index: 2;
            }}
            td {{
                max-width: 1000px;
                word-break: break-word;
            }}
        </style>
        {html_table}
    </div>
    """

    display(HTML(html_text))



# =========================
# 7. 打印 / 展示结果
# =========================

print("========== 汇总统计 ==========")
print(f"品种数量: {len(result_ordered)}")
print(f"发生过交易时间规则变动的品种数量: {len(changed_ordered)}")
print(f"未发生交易时间规则变动的品种数量: {len(unchanged_ordered)}")

display_full_df(
    result_ordered,
    "全部品种：交易时间规则时间线",
    max_height=700,
    min_width=1600,
)

display_full_df(
    changed_ordered,
    "有交易时间规则变动的品种：按先后顺序",
    max_height=700,
    min_width=1600,
)

display_full_df(
    unchanged_ordered,
    "没有交易时间规则变动的品种",
    max_height=700,
    min_width=1600,
)

display_full_df(
    rule_versions_long,
    "规则长表：一行一套历史交易时间规则",
    max_height=700,
    min_width=1700,
)


# =========================
# 8. 导出完整 HTML
# =========================

def build_html_section(df, title):
    show_df = prepare_html_df(df)

    return f"""
    <h2>{title}</h2>
    <div class="table-box">
        {show_df.to_html(index=False, escape=False)}
    </div>
    """


html_doc = f"""
<!DOCTYPE html>
<html lang="zh-CN">
<head>
<meta charset="utf-8">
<title>期货品种交易时间规则变动表</title>
<style>
body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Microsoft YaHei", Arial, sans-serif;
    margin: 24px;
    background: #111;
    color: #eee;
}}

h1, h2 {{
    color: #fff;
}}

.table-box {{
    max-height: 720px;
    overflow: auto;
    border: 1px solid #555;
    margin-bottom: 32px;
}}

table {{
    border-collapse: collapse;
    font-size: 12px;
    min-width: 1600px;
}}

th, td {{
    border: 1px solid #555;
    padding: 4px 8px;
    vertical-align: top;
    white-space: normal;
}}

th {{
    position: sticky;
    top: 0;
    background: #222;
    color: #fff;
    z-index: 2;
}}

td {{
    max-width: 1000px;
    word-break: break-word;
}}
</style>
</head>
<body>

<h1>期货品种交易时间规则变动表</h1>

<p>品种数量：{len(result_ordered)}</p>
<p>发生过交易时间规则变动的品种数量：{len(changed_ordered)}</p>
<p>未发生交易时间规则变动的品种数量：{len(unchanged_ordered)}</p>

{build_html_section(result_ordered, "全部品种：交易时间规则时间线")}

{build_html_section(changed_ordered, "有交易时间规则变动的品种：按先后顺序")}

{build_html_section(unchanged_ordered, "没有交易时间规则变动的品种")}

{build_html_section(rule_versions_long, "规则长表：一行一套历史交易时间规则")}

</body>
</html>
"""

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_doc)

print(f"已导出完整 HTML: {OUTPUT_HTML}")

获取到期货合约数量: 10579
成功获取合约信息数量: 10579
失败 batch 数量: 0
合约-规则段明细行数: 11354
========== 汇总统计 ==========
品种数量: 95
发生过交易时间规则变动的品种数量: 43
未发生交易时间规则变动的品种数量: 52


exchange,exchange_name,variety,changed_emoji,changed,n_session_rules,n_contracts,rule_timeline_text
CCFX,中金所,IC,🔴 已变动,True,2,141,规则1：2015-04-16 ~ 2015-12-31；09:15~11:30 | 13:00~15:15\n规则2：2016-01-04 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,IF,🔴 已变动,True,2,201,规则1：2010-04-16 ~ 2015-12-31；09:15~11:30 | 13:00~15:15\n规则2：2016-01-04 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,IH,🔴 已变动,True,2,141,规则1：2015-04-16 ~ 2015-12-31；09:15~11:30 | 13:00~15:15\n规则2：2016-01-04 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,T,🔴 已变动,True,2,49,规则1：2015-03-20 ~ 2020-07-17；09:15~11:30 | 13:00~15:15\n规则2：2020-07-20 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
CCFX,中金所,TF,🔴 已变动,True,2,59,规则1：2012-06-11 ~ 2020-07-17；09:15~11:30 | 13:00~15:15\n规则2：2020-07-20 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
CCFX,中金所,TS,🔴 已变动,True,2,36,规则1：2018-08-17 ~ 2020-07-17；09:15~11:30 | 13:00~15:15\n规则2：2020-07-20 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
XDCE,大商所,A,🔴 已变动,True,4,138,规则1：2003-07-15 ~ 2014-12-26；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2014-12-27 ~ 2015-05-08；21:00~02:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则3：2015-05-09 ~ 2019-03-29；21:00~23:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则4：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,B,🔴 已变动,True,4,192,规则1：2004-12-22 ~ 2014-12-26；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2014-12-27 ~ 2015-05-08；21:00~02:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则3：2015-05-09 ~ 2019-03-29；21:00~23:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则4：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,C,🔴 已变动,True,2,138,规则1：2004-09-22 ~ 2019-03-29；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,CS,🔴 已变动,True,2,77,规则1：2014-12-19 ~ 2019-03-29；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00


exchange,exchange_name,variety,changed_emoji,changed,n_session_rules,n_contracts,rule_timeline_text
CCFX,中金所,IC,🔴 已变动,True,2,141,规则1：2015-04-16 ~ 2015-12-31；09:15~11:30 | 13:00~15:15\n规则2：2016-01-04 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,IF,🔴 已变动,True,2,201,规则1：2010-04-16 ~ 2015-12-31；09:15~11:30 | 13:00~15:15\n规则2：2016-01-04 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,IH,🔴 已变动,True,2,141,规则1：2015-04-16 ~ 2015-12-31；09:15~11:30 | 13:00~15:15\n规则2：2016-01-04 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,T,🔴 已变动,True,2,49,规则1：2015-03-20 ~ 2020-07-17；09:15~11:30 | 13:00~15:15\n规则2：2020-07-20 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
CCFX,中金所,TF,🔴 已变动,True,2,59,规则1：2012-06-11 ~ 2020-07-17；09:15~11:30 | 13:00~15:15\n规则2：2020-07-20 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
CCFX,中金所,TS,🔴 已变动,True,2,36,规则1：2018-08-17 ~ 2020-07-17；09:15~11:30 | 13:00~15:15\n规则2：2020-07-20 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
XDCE,大商所,A,🔴 已变动,True,4,138,规则1：2003-07-15 ~ 2014-12-26；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2014-12-27 ~ 2015-05-08；21:00~02:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则3：2015-05-09 ~ 2019-03-29；21:00~23:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则4：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,B,🔴 已变动,True,4,192,规则1：2004-12-22 ~ 2014-12-26；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2014-12-27 ~ 2015-05-08；21:00~02:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则3：2015-05-09 ~ 2019-03-29；21:00~23:30 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则4：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,C,🔴 已变动,True,2,138,规则1：2004-09-22 ~ 2019-03-29；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,CS,🔴 已变动,True,2,77,规则1：2014-12-19 ~ 2019-03-29；09:00~10:15 | 10:30~11:30 | 13:30~15:00\n规则2：2019-03-30 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00


exchange,exchange_name,variety,changed_emoji,changed,n_session_rules,n_contracts,rule_timeline_text
CCFX,中金所,IM,🟢 未变动,False,1,54,规则1：2022-07-22 ~ 2200-01-01；09:30~11:30 | 13:00~15:00
CCFX,中金所,TL,🟢 未变动,False,1,18,规则1：2023-04-21 ~ 2200-01-01；09:30~11:30 | 13:00~15:15
GFEX,广期所,LC,🟢 未变动,False,1,45,规则1：2023-07-21 ~ 2200-01-01；09:00~10:15 | 10:30~11:30 | 13:30~15:00
GFEX,广期所,PD,🟢 未变动,False,1,9,规则1：2025-11-27 ~ 2200-01-01；09:00~10:15 | 10:30~11:30 | 13:30~15:00
GFEX,广期所,PS,🟢 未变动,False,1,28,规则1：2024-12-26 ~ 2200-01-01；09:00~10:15 | 10:30~11:30 | 13:30~15:00
GFEX,广期所,PT,🟢 未变动,False,1,9,规则1：2025-11-27 ~ 2200-01-01；09:00~10:15 | 10:30~11:30 | 13:30~15:00
GFEX,广期所,SI,🟢 未变动,False,1,50,规则1：2022-12-22 ~ 2200-01-01；09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,BB,🟢 未变动,False,1,147,规则1：2013-12-06 ~ 2200-01-01；09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,BZ,🟢 未变动,False,1,19,规则1：2025-07-08 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00
XDCE,大商所,EB,🟢 未变动,False,1,90,规则1：2019-09-26 ~ 2200-01-01；21:00~23:00 | 09:00~10:15 | 10:30~11:30 | 13:30~15:00


exchange,exchange_name,variety,rule_no,first_start_date,last_end_date,sessions_text,n_contracts_with_rule,example_contracts
CCFX,中金所,IC,1,2015-04-16,2015-12-31,09:15~11:30 | 13:00~15:15,14,"IC1505.CCFX, IC1506.CCFX, IC1507.CCFX, IC1508.CCFX, IC1509.CCFX, IC1510.CCFX, IC1511.CCFX, IC1512.CCFX"
CCFX,中金所,IC,2,2016-01-04,2200-01-01,09:30~11:30 | 13:00~15:00,133,"IC1601.CCFX, IC1602.CCFX, IC1603.CCFX, IC1604.CCFX, IC1605.CCFX, IC1606.CCFX, IC1607.CCFX, IC1608.CCFX"
CCFX,中金所,IF,1,2010-04-16,2015-12-31,09:15~11:30 | 13:00~15:15,74,"IF1005.CCFX, IF1006.CCFX, IF1007.CCFX, IF1008.CCFX, IF1009.CCFX, IF1010.CCFX, IF1011.CCFX, IF1012.CCFX"
CCFX,中金所,IF,2,2016-01-04,2200-01-01,09:30~11:30 | 13:00~15:00,133,"IF1601.CCFX, IF1602.CCFX, IF1603.CCFX, IF1604.CCFX, IF1605.CCFX, IF1606.CCFX, IF1607.CCFX, IF1608.CCFX"
CCFX,中金所,IH,1,2015-04-16,2015-12-31,09:15~11:30 | 13:00~15:15,14,"IH1505.CCFX, IH1506.CCFX, IH1507.CCFX, IH1508.CCFX, IH1509.CCFX, IH1510.CCFX, IH1511.CCFX, IH1512.CCFX"
CCFX,中金所,IH,2,2016-01-04,2200-01-01,09:30~11:30 | 13:00~15:00,133,"IH1601.CCFX, IH1602.CCFX, IH1603.CCFX, IH1604.CCFX, IH1605.CCFX, IH1606.CCFX, IH1607.CCFX, IH1608.CCFX"
CCFX,中金所,IM,1,2022-07-22,2200-01-01,09:30~11:30 | 13:00~15:00,54,"IM2208.CCFX, IM2209.CCFX, IM2210.CCFX, IM2211.CCFX, IM2212.CCFX, IM2301.CCFX, IM2302.CCFX, IM2303.CCFX"
CCFX,中金所,T,1,2015-03-20,2020-07-17,09:15~11:30 | 13:00~15:15,25,"T1509.CCFX, T1512.CCFX, T1603.CCFX, T1606.CCFX, T1609.CCFX, T1612.CCFX, T1703.CCFX, T1706.CCFX"
CCFX,中金所,T,2,2020-07-20,2200-01-01,09:30~11:30 | 13:00~15:15,29,"T2009.CCFX, T2012.CCFX, T2103.CCFX, T2106.CCFX, T2109.CCFX, T2112.CCFX, T2203.CCFX, T2206.CCFX"
CCFX,中金所,TF,1,2012-06-11,2020-07-17,09:15~11:30 | 13:00~15:15,35,"TF1303.CCFX, TF1306.CCFX, TF1309.CCFX, TF1312.CCFX, TF1403.CCFX, TF1406.CCFX, TF1409.CCFX, TF1412.CCFX"


已导出完整 HTML: 期货品种交易时间规则变动表_含先后顺序.html


In [27]:
changed_ordered

In [37]:
from jqdatasdk import finance, query

query_day = "2024-01-02"

df = finance.run_query(
    query(finance.FUT_MARGIN)
    .filter(finance.FUT_MARGIN.day == query_day)
    .limit(1000)
)

df

E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)


,id,day,code,exchange,exchange_name,specul_buy_margin_rate,specul_sell_margin_rate,hedg_buy_margin_rate,hedg_sell_margin_rate
0,1692919,2024-01-02,A2401.XDCE,XDCE,大连商品交易所,20.0,20.0,20.0,20.0
1,1692828,2024-01-02,A2403.XDCE,XDCE,大连商品交易所,8.0,8.0,7.0,7.0
2,1692806,2024-01-02,A2405.XDCE,XDCE,大连商品交易所,8.0,8.0,7.0,7.0
3,1692698,2024-01-02,A2407.XDCE,XDCE,大连商品交易所,8.0,8.0,7.0,7.0
4,1692615,2024-01-02,A2409.XDCE,XDCE,大连商品交易所,8.0,8.0,7.0,7.0
5,1692530,2024-01-02,A2411.XDCE,XDCE,大连商品交易所,8.0,8.0,7.0,7.0
6,1692894,2024-01-02,AG2401.XSGE,XSGE,上海期货交易所,15.0,15.0,15.0,15.0
7,1692880,2024-01-02,AG2402.XSGE,XSGE,上海期货交易所,10.0,10.0,10.0,10.0
8,1692845,2024-01-02,AG2403.XSGE,XSGE,上海期货交易所,9.0,9.0,8.0,8.0
9,1692784,2024-01-02,AG2404.XSGE,XSGE,上海期货交易所,9.0,9.0,8.0,8.0


In [38]:
finance.FUT_CHARGE

jqdatasdk.table.FUT_CHARGE

In [40]:
finance.run_query(
    query(finance.FUT_CHARGE)
    .limit(1000)
)

E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)


,id,day,code,exchange,exchange_name,unit,clearance_charge,opening_charge,short_clearance_charge,short_opening_charge
0,84841,2013-01-04,A1301.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
1,84839,2013-01-04,A1303.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
2,84863,2013-01-04,A1305.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
3,84835,2013-01-04,A1307.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
4,84848,2013-01-04,A1309.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
5,84795,2013-01-04,A1311.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
6,84830,2013-01-04,A1401.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
7,84831,2013-01-04,A1403.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
8,84832,2013-01-04,A1405.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0
9,84802,2013-01-04,B1301.XDCE,XDCE,大连商品交易所,元/手,2.00,2.0,2.00,2.0


In [1]:
jq.get_all_securities(types=['futures'], date='2026-07-09')

NameError: name 'jq' is not defined

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import kv as besselk

def generalized_hyperbolic_pdf(x, lam=1, alpha=1, beta=0, delta=1, mu=0):
    """
    广义双曲分布(GHD)概率密度函数，详见Barndorff-Nielsen (1977), "Exponentially decreasing distributions for the logarithm of particle size".

    参数:
        x: 自变量
        lam: 形状参数 (lambda)
        alpha: 尖峰参数 (> |beta|)
        beta: 偏斜参数
        delta: 尺度参数 (> 0)
        mu: 均值参数
    """
    x = np.asarray(x)
    z = np.sqrt(delta**2 + (x - mu)**2)
    part1 = (alpha**2 - beta**2) ** (lam/2)
    part2 = (z/delta) ** (lam - 0.5)
    part3 = besselk(lam - 0.5, alpha * z)
    part4 = np.exp(beta * (x - mu))
    denom = (np.sqrt(2*np.pi) * delta**(0.5-lam) * besselk(lam, delta * np.sqrt(alpha**2 - beta**2)))
    pdf = part1 * part2 * part3 * part4 / denom
    return pdf

def plot_generalized_hyperbolic(lam=1, alpha=1.5, beta=0, delta=1, mu=0, xrange=(-6,6), num=1000):
    xs = np.linspace(xrange[0], xrange[1], num)
    ys = generalized_hyperbolic_pdf(xs, lam=lam, alpha=alpha, beta=beta, delta=delta, mu=mu)
    plt.figure(figsize=(8,4))
    plt.plot(xs, ys, label=f'GHD($\\lambda$={lam}, $\\alpha$={alpha}, $\\beta$={beta}, $\\delta$={delta}, $\\mu$={mu})')
    plt.title("广义双曲分布概率密度函数")
    plt.xlabel("x")
    plt.ylabel("PDF")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

plot_generalized_hyperbolic()